In [10]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from src import content_based

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.split import temporal_split
from src.hybrid import hybrid_scores, top_n, normalize
from src import evaluation as ev

ratings = pd.read_csv(ROOT / "data" / "raw" / "ratings.csv")
movies = pd.read_csv(ROOT / "data" / "raw" / "movies.csv")
train, val, test = temporal_split(ratings)
print(train.shape, val.shape, test.shape)

(80896, 4) (10122, 4) (9818, 4)


In [11]:
K = 10
LIKE = 4.0

relevant_by_user = {
    u: set(g.loc[g["rating"] >= LIKE, "movieId"]) for u, g in val.groupby("userId")
}
relevant_by_user = {u: s for u, s in relevant_by_user.items() if s}
seen_by_user = train.groupby("userId")["movieId"].apply(set).to_dict()
all_movies = sorted(movies["movieId"].unique())

print("Utilisateurs évalués :", len(relevant_by_user))

Utilisateurs évalués : 565


In [12]:
relevant_by_user = dict(list(relevant_by_user.items())[:100])

In [13]:
C = train["rating"].mean()
m = 50
stats = train.groupby("movieId")["rating"].agg(["count", "mean"])
stats["score"] = (stats["count"] / (stats["count"] + m)) * stats["mean"] \
               + (m / (stats["count"] + m)) * C
pop_scores = stats["score"].to_dict()


def popularity_scores(user_id, movie_ids):
    return {mid: pop_scores.get(mid, C) for mid in movie_ids}


def random_scores(user_id, movie_ids):
    rng = np.random.default_rng(user_id)
    return dict(zip(movie_ids, rng.random(len(movie_ids))))

In [17]:
import time
t = time.time()
results, recs = evaluate(popularity_scores, content_based.get_scores, alphas)
print("Durée :", round(time.time() - t, 1), "s")
results

Durée : 22.1 s


,alpha,P@10,R@10,NDCG@10
0,0.0,0.001,0.000714,0.000734
1,0.1,0.003,0.004325,0.005395
2,0.2,0.009,0.021087,0.015306
3,0.3,0.011,0.024572,0.018460
4,0.4,0.013,0.024048,0.020807
5,0.5,0.022,0.045306,0.032263
6,0.6,0.033,0.058160,0.047071
7,0.7,0.037,0.056602,0.051743
8,0.8,0.042,0.070010,0.066771
9,0.9,0.041,0.067914,0.064024


## juste une petite interpretation

si sik alpha = 0 --> les genre apportnt de l'information
si = 1 normal les genres seules sont un signal faible 
le maximun de alpha intermidiaire --> la resulta que l hybride cherche 

In [15]:
def evaluate(collab_fn, content_fn, alphas):
    recs = {a: {} for a in alphas}
    for u in relevant_by_user:
        candidates = [mid for mid in all_movies if mid not in seen_by_user.get(u, set())]
        collab = normalize(collab_fn(u, candidates))
        content = normalize(content_fn(u, candidates))
        for a in alphas:
            scores = hybrid_scores(collab, content, a)
            recs[a][u] = [mid for mid, _ in top_n(scores, K)]

    rows = []
    for a in alphas:
        rows.append({
            "alpha": a,
            "P@10": ev.mean_at_k(ev.precision_at_k, recs[a], relevant_by_user, K),
            "R@10": ev.mean_at_k(ev.recall_at_k, recs[a], relevant_by_user, K),
            "NDCG@10": ev.mean_at_k(ev.ndcg_at_k, recs[a], relevant_by_user, K),
        })
    return pd.DataFrame(rows), recs


alphas = [round(a, 1) for a in np.arange(0, 1.01, 0.1)]
results, recs = evaluate(popularity_scores, random_scores, alphas)
results

,alpha,P@10,R@10,NDCG@10
0,0.0,0.000,0.000000,0.000000
1,0.1,0.003,0.003076,0.006616
2,0.2,0.008,0.014576,0.014151
3,0.3,0.009,0.016076,0.017755
4,0.4,0.014,0.021838,0.024200
5,0.5,0.020,0.031386,0.030331
6,0.6,0.022,0.043386,0.037277
7,0.7,0.026,0.047824,0.048154
8,0.8,0.029,0.051042,0.051797
9,0.9,0.035,0.061177,0.058445


In [16]:
best = results.loc[results["NDCG@10"].idxmax()]
print("Meilleur alpha (Validation, NDCG@10) :", best["alpha"])

best_recs = recs[best["alpha"]]
coverage = len({mid for l in best_recs.values() for mid in l}) / len(all_movies)
print(f"Couverture du catalogue : {coverage:.1%}")

Meilleur alpha (Validation, NDCG@10) : 1.0
Couverture du catalogue : 0.5%


**Interprétation (baselines)**

- Aléatoire seul (alpha = 0) : NDCG@10 ≈ 0,001. C'est le plancher.
- Popularité seule (alpha = 1) : P@10 = 0,034, R@10 = 0,055, NDCG@10 = 0,054. C'est le baseline à battre.
- Ces métriques augmentent régulièrement avec alpha, ce qui valide le pipeline d'évaluation.
- Couverture du catalogue de la popularité : 0,7 % (environ 68 films recommandés à tous les utilisateurs). Un modèle personnalisé doit viser une couverture bien plus élevée.
- Le côté content est ici un score aléatoire : le meilleur alpha sera recalculé avec les vrais modèles.